# Game Store Management System
## Overview
### Functionalities

The game store management system includes:

1. **Game search**
2. **Game rent**
3. **Game return**
4. **In-Store Booking**
5. **Inventory Pruning**

*Instructions:*
*   Use the **Game search** section to search for board games or video games.
  Select the game type and enter a title or genre in the search box to display matching games. (Partial matches are also allowed)

* To rent a game, use the **Rent** section.
  Select a valid customer ID and an available game ID from the drop-down menus, then click **Rent Game**. The system will confirm whether the rental was successful or explain why it could not proceed.

* To return a game, use the **Return** section.
  Select the game ID, choose a rating using the slider, and optionally enter a comment. Click **Return Game** to complete the return. The **Clear** button can be used to reset the return fields.

* To book an in-store session, use the **Booking** section.
  Select a customer ID, confirm the booking date, choose a time slot (2pm or 6pm), and select the number of guests (0–3).

* To return a game, use the Return section. Select the game ID, choose a rating using the slider, and optionally enter a comment.

* In order for the menu to load properly load all cells and then return to the menu cell





# Database Module (`database`)

## Overview
This section contains functions used by the other sections to interact with the text-file database (read, search, update, append).

## Data Files Used
- **Board_Game_Info.txt**: stores board game records
- **Video_Game_Info.txt**: stores video game records
- **Rental.txt**: stores rental history
- **Booking.txt**: stores booking history
- **Game_Feedback.txt stores the game feedback
-** Subscription_Info.txt stores the information regarding each customers subscription
- **Game_Feedback.txt**: populated via `feedbackManager.pyc`
`subscriptionManager.pyc`

## Notes

- Dates are stored as `YYYY-MM-DD`
- A return date of `NULL` means the game is currently rented


In [ ]:
#database
from google.colab import files
Uploaded = files.upload()

Saving Board_Game_Info.txt to Board_Game_Info (1).txt
Saving Booking.txt to Booking (1).txt
Saving feedbackManager.pyc to feedbackManager (1).pyc
Saving Game_Feedback.txt to Game_Feedback (1).txt
Saving Rental.txt to Rental (1).txt
Saving Subscription_Info.txt to Subscription_Info (1).txt
Saving subscriptionManager.pyc to subscriptionManager (1).pyc
Saving Video_Game_Info.txt to Video_Game_Info (1).txt


# In-Store Booking (`booking`)

## Overview
This section allows subscribers to book face-to-face gaming sessions at the store. Sessions are available between 2pm–6pm and 6pm–10pm, with a maximum capacity of 50 people per session.

## What the system checks
- The customer has an active subscription
- The selected time slot is valid
- The number of guests is within the allowed limit
- The total number of people booked does not exceed capacity




In [ ]:
#gameBooking
import subscriptionManager as sm

booking_file = "Booking.txt"
valid_slots = ["2pm", "6pm"]
max_cap = 50
subscriptions = sm.load_subscriptions()


def check_booking_validity(customer_ID):
  customer_ID = customer_ID.strip()
  return sm.check_subscription(customer_ID, subscriptions)
#Check to see if the customer has a valid subscription

def check_booking_availability(slot):
  if slot in valid_slots:
    return True
  else:
    return False
#Check to see if the timeslot for the booking is available

def guest_check(guest):
  if guest < 0 or guest > 3:
    return False
  else:
    return True
#Checks to see if the amount of guest is valid

def check_date_validity(booking_date):
  parts = booking_date.split("-")
  return len(parts) == 3
#Checks if a valid date has been entered

def current_people_in_slot(slot, booking_date):
  total = 0
  with open(booking_file, "r") as f:
    next(f)
    for line in f:
      parts = line.strip()
      sections = parts.split(",")

      file_date = sections[1].strip()
      file_slot = sections[2].strip()
      file_guest = sections[3].strip()

      if file_slot == slot and file_date == booking_date:
        total = total + 1 + int(file_guest)

  return total
#Total people already booked

def add_booking(customer_ID, slot, date, guests):
  customer_ID = customer_ID.strip()
  slot = slot.strip()

  new_booking = customer_ID + "," + date + "," + slot + "," + str(guests) + "\n"
  with open(booking_file, "a") as f:
    f.write(new_booking)

  return "SUCCESS"
#Adds a new booking to the file

def book_session(customer_ID, slot, date, guests):
  if check_booking_validity(customer_ID) == False:
    return "INVALID_SUB"
  if check_date_validity(date) == False:
    return "INVALID_DATE"
  if guest_check(guests) == False:
    return "INVALID_GUESTS"
  if check_booking_availability(slot) == False:
    return "INVALID_SLOT"

  current_total = current_people_in_slot(slot, date)
  new_people = 1 + guests

  if current_total + new_people > max_cap:
    return "SLOT_FULL"

  add_booking(customer_ID, slot, date, guests)
  return "SUCCESS"
#Adds the booking if valid


# Menu (`menu`)

# Overview
The menu provides the main interface for the Game Store Management System. It brings together all core functionalities into a single, easy-to-use window.

## What the menu allows
The menu allows the store manager to:
- Search for board and video games
- Rent games to subscribed customers
- Return rented games and collect feedback
- Book in-store gaming sessions
- View inventory insights and pruning suggestions

## How it works
Each section of the menu is clearly separated and contains the necessary input fields and buttons to perform its function. Drop-down menus and sliders are used where possible to reduce input errors and improve usability.

In [ ]:
#menu
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from datetime import date

customer_dropdown = widgets.Dropdown(description="Customer ID:")
game_dropdown = widgets.Dropdown(description="Game ID:")
#Creates a dropdown menu

choice_buttons = widgets.ToggleButtons(
    options = [('Board games', 'B'), ('Video games', 'V')],
    description = 'Game type')

search_box = widgets.Text(
    description = 'Search',
    placeholder = 'What genre or Title are you looking for')

rent_button = widgets.Button(
    description="Rent Game",
    button_style="success"
)

return_game_dropdown = widgets.Dropdown(
    description="Game ID:",
    options=[]
  )

return_rating_slider = widgets.IntSlider(
    description="Rating:",
    min=1, max=5, step=1,
    value=5)

return_comment_box = widgets.Text(
    description="Comment:",
    placeholder="Enter comment (optional)"
  )

return_button = widgets.Button(
    description="Return Game",
    button_style="warning"
  )
clear_return_button = widgets.Button(
    description="Clear",
    button_style=""
  )

booking_customer_dropdown = widgets.Dropdown(
    options=sorted(list(subscriptions.keys())),
    description="Customer:"
)

booking_date_box = widgets.Text(
    value=date.today().isoformat(),
    description="Date:",
    placeholder="YYYY-MM-DD"
)

booking_slot_dropdown = widgets.Dropdown(
    options=valid_slots,
    description="Slot:"
)

booking_guest_slider = widgets.IntSlider(
    value=0, min=0, max=3, step=1,
    description="Guests:"
)

booking_button = widgets.Button(
    description="Book Session",
    button_style="info"
)

inventory_bottom3_btn = widgets.Button(
    description="Bottom 3",
    button_style="primary"
)

inventory_avg_chart_btn = widgets.Button(
    description="Average ratings",
    button_style="info"
)

inventory_rental_chart_btn = widgets.Button(
    description="Rental genre chart",
    button_style="info"
)

output_box = widgets.Output()

def on_book_clicked(b):
  output_box.clear_output()

  customer_ID = booking_customer_dropdown.value
  booking_date = booking_date_box.value.strip()
  slot = booking_slot_dropdown.value
  guests = booking_guest_slider.value

  with output_box:
    result = book_session(customer_ID, slot, booking_date, guests)

    if result == "INVALID_SUB":
      display(widgets.HTML("<b style='color:red;'>Customer does not have an active subscription</b>"))
    elif result == "INVALID_DATE":
      display(widgets.HTML("<b style='color:red;'>Invalid date format (use YYYY-MM-DD)</b>"))
    elif result == "INVALID_GUESTS":
      display(widgets.HTML("<b style='color:red;'>Guests must be between 0 and 3</b>"))
    elif result == "INVALID_SLOT":
      display(widgets.HTML("<b style='color:red;'>Invalid slot selected</b>"))
    elif result == "SLOT_FULL":
      display(widgets.HTML("<b style='color:red;'>That slot is full (over 50 capacity)</b>"))
    elif result == "SUCCESS":
      display(widgets.HTML("<b style='color:green;'>Booking confirmed</b>"))

booking_button.on_click(on_book_clicked)

def on_inventory_bottom3_clicked(b):
    output_box.clear_output()
    bottom3 = bottom_rated_games()[:3]

    html = "<b>Bottom 3 games (by avg rating, rating count, then ID)</b><br><br>"

    for game_id, name, rating_count, avg_rating in bottom3:
        html += f"{game_id} - {name} | Ratings: {rating_count} | Avg: {avg_rating:.2f}<br>"

    with output_box:
        display(widgets.HTML(html))

def on_inventory_avg_chart_clicked(b):
    output_box.clear_output()
    with output_box:
        plot_average_ratings()

def on_inventory_rental_chart_clicked(b):
    output_box.clear_output()
    with output_box:
        plot_rental_frequency()

inventory_bottom3_btn.on_click(on_inventory_bottom3_clicked)
inventory_avg_chart_btn.on_click(on_inventory_avg_chart_clicked)
inventory_rental_chart_btn.on_click(on_inventory_rental_chart_clicked)

inventory_section = widgets.VBox([
    widgets.HTML("<hr><b>Inventory pruning</b>"),
    widgets.HBox([
        inventory_bottom3_btn,
        inventory_avg_chart_btn,
        inventory_rental_chart_btn
    ])
])

display(
    widgets.VBox([
        widgets.HTML("<hr><b>Game search</b>"),
        choice_buttons,
        search_box,

        widgets.HTML("<hr><b>Rent</b>"),
        customer_dropdown,
        game_dropdown,
        rent_button,

        widgets.HTML("<hr><b>Return</b>"),
        return_game_dropdown,
        return_rating_slider,
        return_comment_box,
        widgets.HBox([return_button, clear_return_button]),

        widgets.HTML("<hr><b>Booking</b>"),
        booking_customer_dropdown,
        booking_date_box,
        booking_slot_dropdown,
        booking_guest_slider,
        booking_button,

        inventory_section,

        widgets.HTML("<hr>"),
        output_box
    ])
)


#Displays the GUI layout

# Game Search (`gamesearch`)

## Overview
This section allows the store manager to search for both board games and video games using a title or genre. It returns all relevant information for matching games and clearly shows availability.

## What the manager can do
- Search board games or video games
- Enter a keyword related to the game name or genre
- View game details
- See availability based on current rental records

## How it works
The system reads the relevant game inventory file and checks Rental.txt to determine whether a game is currently rented. Games with no active rental are shown as available.

## Output
A list of matching games is displayed, including:
- Game ID
- Name or title
- Genre
- Additional details (players or platform)
- Availability status

In [ ]:
#gameSearch
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def is_game_available(game_id):
    with open("Rental.txt", "r") as f:
      next(f)
      for line in f:
          section = [item.strip() for item in line.split(",")]
          if section[0] == game_id:
              return_date = section[2].strip().lower()
              if return_date == "null" or return_date == "":
                  return False
    return True
#Checks whether the game is available

def show_table(data,columns):
    fig, ax = plt.subplots(figsize=(10, 1.5 + len(data) * 0.35))
    ax.axis('off')
    table = ax.table(cellText=data, colLabels=columns, loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1,1.2)
    plt.tight_layout()
    plt.show()
#Displays the results as a table

def runsearch(_=None):
  output_box.clear_output()
  results = []
  choice = choice_buttons.value
  searched_term = search_box.value.strip()

  if choice == "B":
    filename = "Board_Game_Info.txt"
  else:
    filename = "Video_Game_Info.txt"

  with open(filename, "r") as f:
        next(f)
        for line in f:
            section = [item.strip() for item in line.split(",")]
            if searched_term in section[1].lower() or searched_term in section[3].lower():
                availability = is_game_available(section[0])
                section.append("Available" if availability else "Rented Out")
                results.append(section)
  #Selects where the range from where games are coming from

  with output_box:
    if results:
        columns = ["Game ID", "Name", "No. Players", "Genre", "Purchase date", "Availability"]
        show_table(results, columns)
    else:
      display(widgets.HTML("<b style='color:red;'>No results found</b>"))
#Outputs search results

search_box.on_submit(runsearch)
choice_buttons.observe(lambda change: runsearch(None), names='value')


# Game Rent (`gamerent`)

## Overview
This section is responsible for renting a game to a customer. The store manager selects a customer ID and a game ID using the interface, and the system performs a series of checks to determine whether the rental can proceed.

## How the system works
- Customer IDs are loaded and displayed in a drop-down menu.
- This ensures that only valid customers and existing games can be chosen.

## Checks performed
Before a rental is completed, the system checks that:
- The customer has a valid and active subscription
- The game is not currently rented
- The customer has not exceeded the maximum number of rentals allowed by their subscription type


In [ ]:
#gameRent
import subscriptionManager as sm
from datetime import date
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

subscriptions = sm.load_subscriptions()
customer_dropdown.options = sorted(subscriptions.keys())
#Sorts the customer ID's in order for the dropdown menu

def refresh_game_dropdown(_=None):
  game_ids = []
  choice = choice_buttons.value

  if choice == "B":
    filename = "Board_Game_Info.txt"
  else:
    filename = "Video_Game_Info.txt"

  with open(filename, "r") as f:
      next(f)
      for line in f:
          game_ids.append(line.split(",")[0].strip())

  game_dropdown.options = sorted(game_ids)
#Refreshes the dropdowns

choice_buttons.observe(refresh_game_dropdown, names='value')
refresh_game_dropdown()

def check_rental_amount(customer_ID):
  rental_amount = 0
  with open("Rental.txt", "r") as f:
    next(f)
    for line in f:
      parts = line.strip()
      sections = parts.split(",")
      if sections[3] == customer_ID and sections[2] == "NULL":
        rental_amount = rental_amount + 1
  return rental_amount
#Checks how many rents the customer has

def rental_validity(customer_ID):
  return sm.check_subscription(customer_ID, subscriptions)
#Checks if its valid

def check_availability(game_ID):
  with open("Rental.txt", "r") as f:
    next(f)
    for line in f:
      parts = line.strip()
      sections = parts.split(",")
      if sections[0] == game_ID and sections[2] == "NULL":
        return False
    return True
#Checks availability of game ID

def check_game_ID(game_ID):
  with open("Board_Game_Info.txt", "r") as f:
    next(f)
    for line in f:
      parts = line.strip()
      sections = parts.split(",")
      if sections[0] == game_ID:
          return True

  with open("Video_Game_Info.txt", "r") as f:
    next(f)
    for line in f:
      parts = line.strip()
      sections = parts.split(",")
      if sections[0] == game_ID:
          return True

  return False
#Checks the game ID if its in the record

def rent_game(customer_ID, game_ID):
  if rental_validity(customer_ID) == False:
    return "INVALID_SUB"
  if check_game_ID(game_ID) == False:
    return "INVALID_GAME"
  if check_availability(game_ID) == False:
    return "NOT_AVAILABLE"
  #If the subscription is invalid, ID doesnt exist or game not available then it should stop

  sub_type = subscriptions[customer_ID]["SubscriptionType"]
  allowed = sm.get_rental_limit(sub_type)
  active_rentals = check_rental_amount(customer_ID)

  if active_rentals >= allowed:
    return "LIMIT_REACHED"

  today = date.today().isoformat()
  #Makes sure that the date is in a consistent form

  with open("Rental.txt", "a") as f:
    f.write(f"\n{game_ID},{today},NULL,{customer_ID}")

  return "SUCCESS"

def on_rent_button_clicked(b):
    output_box.clear_output()
    result = rent_game(customer_dropdown.value, game_dropdown.value)

    with output_box:
        if result == "INVALID_SUB":
            display(widgets.HTML("<b style='color:red;'>Invalid subscription</b>"))

        elif result == "INVALID_GAME":
            display(widgets.HTML("<b style='color:red;'>Game ID does not exist</b>"))

        elif result == "NOT_AVAILABLE":
            display(widgets.HTML("<b style='color:red;'>Game is not available</b>"))

        elif result == "LIMIT_REACHED":
            display(widgets.HTML("<b style='color:red;'>Maximum number of rentals reached</b>"))

        elif result == "SUCCESS":
            display(widgets.HTML("<b style='color:green;'>Game rented successfully</b>"))
        #Uses HTML to present the result but not in plain text

rent_button.on_click(on_rent_button_clicked)


# Game Return (`gamereturn`)

## Overview
This section allows the store manager to return a rented game and collect customer feedback. It updates the rental record and stores feedback using the provided feedback manager.

## What the manager provides
- Game ID
- Rating
- Optional comment

In [ ]:
#gameReturn
import ipywidgets as widgets
from IPython.display import display, clear_output
import feedbackManager as fm
from datetime import date

def check_game_ID(game_ID):
  with open("Board_Game_Info.txt", "r") as f:
    next(f)
    for line in f:
      parts = line.strip()
      sections = parts.split(",")
      if sections[0] == game_ID:
          return True

  with open("Video_Game_Info.txt", "r") as f:
    next(f)
    for line in f:
      parts = line.strip()
      sections = parts.split(",")
      if sections[0] == game_ID:
          return True
  return False
  #Checks if the game ID exist

def check_return_possible(game_ID):
  with open("Rental.txt", "r") as f:
    next(f)
    for line in f:
      part = line.strip()
      sections = part.split(",")
      if sections[0] == game_ID and sections[2] == "NULL":
        return True
  return False
  #Checks if the game is currently rented

def return_game(game_ID, ratingValue, commentValue):
  if check_game_ID(game_ID) == False:
    return "INVALID_GAME"
  if check_return_possible(game_ID) == False:
    return "NOT_RENTED"
  if ratingValue not in [1,2,3,4,5]:
    return "INVALID_RATING"

  today = date.today().isoformat()

  with open("Rental.txt", "r") as f:
    lines = f.readlines()

  updated = False

  for i in range(1, len(lines)):
    line = lines[i].strip()
    sections = line.split(",")

    if sections[0] == game_ID and sections[2] == "NULL":
      sections[2] = today
      lines[i] = ",".join(sections) + "\n"
      updated = True

  if updated == False:
    return "NOT_RENTED"

  with open("Rental.txt", "w") as f:
    f.writelines(lines)

  fm.add_feedback(game_ID, ratingValue, commentValue, "Game_Feedback.txt")
  return "SUCCESS"
  #Returns the game and recording the feedback that has been given

def refresh_return_game_dropdown():
  rented_ids = []
  with open("Rental.txt", "r") as f:
        next(f)
        for line in f:
            parts = line.strip()
            sections = parts.split(",")
            if len(sections) >= 3 and sections[2] == "NULL":
                rented_ids.append(sections[0])

  return_game_dropdown.options = sorted(set(rented_ids))

refresh_return_game_dropdown()
#Update dropdown with games that are currently rented

def on_return_button_clicked(b):
  output_box.clear_output()

  game_id = return_game_dropdown.value
  rating = return_rating_slider.value
  comment = return_comment_box.value.strip()

  with output_box:
    if game_id is None:
      display(widgets.HTML("<b style='color:red;'>Please select a Game ID</b>"))
      return

    result = return_game(game_id, rating, comment)
    if result == "INVALID_GAME":
      display(widgets.HTML("<b style='color:red;'>Game ID does not exist</b>"))
    elif result == "NOT_RENTED":
      display(widgets.HTML("<b style='color:red;'>Game is not currently rented</b>"))
    elif result == "INVALID_RATING":
      display(widgets.HTML("<b style='color:red;'>Rating must be between 1 and 5</b>"))
    elif result == "SUCCESS":
      display(widgets.HTML("<b style='color:green;'>Game returned successfully</b>"))
      return_rating_slider.value = 5
      return_comment_box.value = ""
      refresh_return_game_dropdown()
#Handles the returning of games

def on_clear_return_clicked(b):
  output_box.clear_output()
  return_rating_slider.value = 5
  return_comment_box.value = ""
  with output_box:
    display(widgets.HTML("<b>Cleared</b>"))
#Clears the returns

return_button.on_click(on_return_button_clicked)
clear_return_button.on_click(on_clear_return_clicked)


# Inventory Pruning (`inverntorypruning`)

## Overview
This section helps identify unpopular games that may be removed from the inventory. Popularity is measured using rental frequency data and game ratings.

## What the manager sees
- A list of games ranked
- Suggestions for which games could be removed the worst 3 games
- A visual representation (chart) showing popularity trends by genre and popularity


In [ ]:
#Inventory Pruning
import matplotlib.pyplot as plt

def load_inventory():
  inventory = {}
  with open("Board_Game_Info.txt", "r") as f:
    next(f)
    for line in f:
      parts = line.strip()
      sections = parts.split(",")

      game_id = sections[0]
      name = sections[1]
      no_players = sections[2]
      genre = sections[3]
      purchase_date = sections[4]

      inventory[game_id] = [name, no_players, genre, purchase_date]

  with open("Video_Game_Info.txt", "r") as f:
    next(f)
    for line in f:
      parts = line.strip()
      sections = parts.split(",")

      game_id = sections[0]
      name = sections[1]
      no_players = sections[2]
      genre = sections[3]
      purchase_date = sections[4]

      inventory[game_id] = [name, no_players, genre, purchase_date]

  return inventory
  #Loads the inventory of games (video games and board games)

def rental_freq_count():
  rental_freq = {}
  with open("Rental.txt", "r") as f:
    next(f)
    for line in f:
      parts = line.strip()
      sections = parts.split(",")

      game_id = sections[0]

      if game_id in rental_freq:
        rental_freq[game_id] = rental_freq[game_id] + 1
      else:
        rental_freq[game_id] = 1

  inventory = load_inventory()

  for game_id in inventory:
    if game_id not in rental_freq:
      rental_freq[game_id] = 0

  return rental_freq
  #Determines the frequency of the games return

def feedback_count():
  feedback_count = {}
  with open ("Game_Feedback.txt", "r") as f:
    next(f)
    for line in f:
      parts = line.strip()
      sections = parts.split(",")

      game_id = sections[0]
      rating = int(sections[1])

      if game_id in feedback_count:
        feedback_count[game_id][0] = feedback_count[game_id][0] + 1
        feedback_count[game_id][1] = feedback_count[game_id][1] + rating
      else:
        feedback_count[game_id] = [1, rating]

  return feedback_count
  #Counts how many ratings each game has received and the total of those ratings

def average_ratings():
  average_ratings = {}

  count = feedback_count()
  inventory = load_inventory()

  for game_id in inventory:
    if game_id in count:
      rating_count = count[game_id][0]
      total_rating = count[game_id][1]

      average = total_rating / rating_count
      average_ratings[game_id] = average
    else:
      average_ratings[game_id] = 0

  return average_ratings
#Calculates the average rating

def bottom_rated_games():
  inventory = load_inventory()
  feedback = feedback_count()
  average = average_ratings()

  rows = []

  for game_id in inventory:
    name = inventory[game_id][0]

    if game_id in feedback:
      rating_count = feedback[game_id][0]
    else:
      rating_count = 0

    avg_rating = average[game_id]
    rows.append([game_id, name, rating_count, avg_rating])

  rows.sort(key=lambda game: (game[3], game[2], game[0]))
  return rows
#Selects the bottom 3 rated games

def plot_average_ratings():
  averages = average_ratings()
  inventory = load_inventory()

  game_names = []
  ratings = []

  for game_id in inventory:
    name = inventory[game_id][0]
    game_names.append(name)

    avg = averages[game_id]
    ratings.append(avg)

  plt.figure(figsize=(10, 6))
  plt.bar(game_names, ratings)
  plt.xlabel("Game")
  plt.ylabel("Average Rating")
  plt.title("Average Ratings of Games")
  plt.xticks(rotation=90)
  plt.tight_layout()
  plt.ylim(0, 5)

  plt.show()
#Plots the game and the average game rating

def plot_rental_frequency():
  inventory = load_inventory()
  rental_freq = rental_freq_count()

  genre_counts = {}

  for game_id in inventory:
    genre = inventory[game_id][2]
    rental_count = rental_freq[game_id]

    if genre in genre_counts:
      genre_counts[genre] = genre_counts[genre] + rental_count
    else:
      genre_counts[genre] = rental_count

  genres = list(genre_counts.keys())
  rental_counts = list(genre_counts.values())

  plt.figure(figsize=(8, 8))
  plt.pie(rental_counts, labels=genres, autopct='%1.0f%%', startangle=180)
  plt.title("Proportion of Rentals by Genre")
  plt.tight_layout()

  plt.show()
  #Plot the rental frequency by genre

# `5 security issues`

1. The system assumes that the interface is always used by an authorised store manager. The lack of a login mechanism or role-based access control means that an unauthorised user could rent games, return games, and/or modify bookings if they gain access to the notebook.

2. Although the use of dropdown menus reduces the likelihood of incorrect input, errors may still occur. A malformed or manually edited input may result in unexpected behaviour or unintended database updates.

3. All customer, rental, and booking data is stored in plain text files. If an individual gains access to these files, they can easily read, modify, or delete the data. Sensitive information, such as customer IDs or rental history, is not encrypted or access-restricted in any way.

4. The system makes use of external compiled modules (`subscriptionManager.pyc` and `feedbackManager.pcy`). As their source code cannot be reviewed internally, the system must assume that they handle data correctly and securely. Any flaws or vulnerabilities within these modules could negatively impact the overall system.

5. The system also assumes that all text files are correctly structured (for example, correct column order, headers present, and NULL used for active rentals). If a file isn't the system may misinterpret data as the system’s correct functioning relies on trust in the file format.
